In [17]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.callbacks import LambdaCallback, EarlyStopping, ModelCheckpoint

import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import regularizers
from tensorflow.keras.layers import Input, Dropout, Dense, Layer, Embedding, Lambda
from keras_hub.layers import PositionEmbedding
from tensorflow.keras.layers import Embedding, MultiHeadAttention, LayerNormalization, GlobalMaxPooling1D, GlobalAveragePooling1D, TextVectorization, BatchNormalization
from tensorflow.keras.models import Model, Sequential

import yfinance as yf
from config import config

from dataclasses import dataclass
import glob
from pprint import pprint

In [21]:
# Get the dataset
# The input is string of news headline and output is the corresponding sentiment
sentiment_raw = pd.read_csv(os.path.join("data", "sentiment.csv"), encoding='latin1', header = None)
sentiment_raw.columns = ["Output", "Input"]
sentiment_dataset = sentiment_raw[["Input", "Output"]]

X, y = sentiment_dataset['Input'], sentiment_dataset['Output']

# Encode Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Encode Input
tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X)
sequences = tokenizer.texts_to_sequences(X)
X_seq = pad_sequences(sequences, maxlen=50, padding="post")

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_encoded, test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

In [22]:
def save_model_weights(model, file_name, folder_name):
    os.makedirs(folder_name, exist_ok=True)

    words = file_name.split(".")

    model_name = words[0]

    existing_files = [f for f in os.listdir(folder_name) if f.startswith(model_name)]
    next_number = len(existing_files) + 1
    words[0] = f"{model_name}_{next_number}"

    file_name = ".".join(words)
    save_path = os.path.join(folder_name, file_name)

    model.save_weights(save_path)
    print(f"model saved to: {save_path}")

In [23]:
# Create Transformer Layer
class MultiHeadSelfAttention(Layer): 
    def __init__(self, embed_dim, num_heads = 8): 
        super(MultiHeadSelfAttention, self).__init__() 
        self.embed_dim = embed_dim 
        self.num_heads = num_heads 
        self.projection_dim = embed_dim // num_heads 
        self.query_dense = Dense(embed_dim) 
        self.key_dense = Dense(embed_dim) 
        self.value_dense = Dense(embed_dim) 
        self.combine_heads = Dense(embed_dim) 
 
    def attention(self, query, key, value): 
        score = tf.matmul(query, key, transpose_b = True) 
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32) 
        scaled_score = score / tf.math.sqrt(dim_key) 
        weights = tf.nn.softmax(scaled_score, axis = -1) 
        output = tf.matmul(weights, value) 
        return output, weights 

    def split_heads(self, x, batch_size): 
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.projection_dim)) 
        return tf.transpose(x, perm=[0, 2, 1, 3]) 

    def call(self, inputs): 
        batch_size = tf.shape(inputs)[0] 
        query = self.query_dense(inputs) 
        key = self.key_dense(inputs) 
        value = self.value_dense(inputs) 
        query = self.split_heads(query, batch_size) 
        key = self.split_heads(key, batch_size) 
        value = self.split_heads(value, batch_size) 
        attention, _ = self.attention(query, key, value) 
        attention = tf.transpose(attention, perm=[0, 2, 1, 3]) 
        concat_attention = tf.reshape(attention, (batch_size, -1, self.embed_dim)) 
        output = self.combine_heads(concat_attention) 
        return output 

class TransformerBlock(Layer): 
    def __init__(self, embed_dim, num_heads, ff_dim, rate = 0.1): 
        super(TransformerBlock, self).__init__() 
        self.att = MultiHeadSelfAttention(embed_dim, num_heads) 
        self.ffn = tf.keras.Sequential([ 
            Dense(ff_dim, activation="relu"), 
            Dense(embed_dim), 
        ]) 

        self.layernorm1 = LayerNormalization(epsilon = 1e-6) 
        self.layernorm2 = LayerNormalization(epsilon = 1e-6) 
        self.dropout1 = Dropout(rate) 
        self.dropout2 = Dropout(rate) 
 

    def call(self, inputs, training): 
        attn_output = self.att(inputs) 
        attn_output = self.dropout1(attn_output, training=training) 
        out1 = self.layernorm1(inputs + attn_output) 
        ffn_output = self.ffn(out1) 
        ffn_output = self.dropout2(ffn_output, training=training) 
        return self.layernorm2(out1 + ffn_output) 
        
class TransformerEncoder(Layer): 
    def __init__(self, num_layers, embed_dim, num_heads, ff_dim, rate = 0.1): 
        super(TransformerEncoder, self).__init__() 
        self.num_layers = num_layers 
        self.embed_dim = embed_dim 
        self.enc_layers = [TransformerBlock(embed_dim, num_heads, ff_dim, rate) for _ in range(num_layers)] 
        self.dropout = Dropout(rate) 

    def call(self, inputs, training=False): 
        x = inputs 
        for i in range(self.num_layers): 
            x = self.enc_layers[i](x, training=training) 
        return x 

In [24]:
# Structure and compile model
max_len = 50
vocab_size = 20000

num_layers = 4
embed_dim = 128
num_heads = 8
ff_dim = 512

input_layer = Input(shape=(max_len,))
x = Embedding(input_dim=vocab_size, output_dim=embed_dim)(input_layer)
x = TransformerEncoder(num_layers, embed_dim, num_heads, ff_dim)(x)
x = Dropout(0.3)(x)
x = GlobalAveragePooling1D()(x)
x = Dropout(0.3)(x)
x = Dense(64, activation="relu", kernel_regularizer=regularizers.l2(0.01))(x)
output_layer = Dense(3, activation="softmax")(x)

model = Model(inputs = input_layer, outputs = output_layer)
model.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics=["accuracy"])

In [25]:
earlyStopping_callback = EarlyStopping(
    monitor="val_accuracy",
    min_delta=0.005,
    patience=5,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=True,
)

In [26]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    verbose=1,
    callbacks=[earlyStopping_callback]
)

Epoch 1/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 18s 123ms/step - accuracy: 0.5629 - loss: 1.6734 - val_accuracy: 0.6005 - val_loss: 1.4395
Epoch 2/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 12s 121ms/step - accuracy: 0.5926 - loss: 1.3401 - val_accuracy: 0.6005 - val_loss: 1.2316
Epoch 3/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 12s 120ms/step - accuracy: 0.5939 - loss: 1.1619 - val_accuracy: 0.6082 - val_loss: 1.0754
Epoch 4/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 12s 127ms/step - accuracy: 0.6861 - loss: 0.9083 - val_accuracy: 0.6559 - val_loss: 0.9366
Epoch 5/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 12s 120ms/step - accuracy: 0.7748 - loss: 0.6814 - val_accuracy: 0.6082 - val_loss: 1.1369
Epoch 6/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 14s 142ms/step - accuracy: 0.8110 - loss: 0.5516 - val_accuracy: 0.6546 - val_loss: 0.9882
Epoch 7/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 14s 142ms/step - accuracy: 0.8323 - loss: 0.4919 - val_accuracy: 0.6379 - val_loss: 1.1497
Epoch 8/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 12s 126ms/step - accuracy: 0.8358 - loss: 0.4587 - val_accu

In [33]:
def predict(text):
    if isinstance(text, str):
        text = [text]
        
    # Tokenize the input
    seq = tokenizer.texts_to_sequences(text)
    padded = pad_sequences(seq, maxlen=50, padding="post")
    
    # Get the Prediction & the Confidence
    pred = model(tf.convert_to_tensor(padded), training=False)
    pred_class = np.argmax(pred, axis=1)
    
    probabilities = tf.nn.softmax(pred, axis=-1)
    confidence_scores = np.max(probabilities, axis=-1)
    
    sentiment = le.inverse_transform(pred_class)
    return sentiment[0], confidence_scores[0]
    
def predict_and_print(text):
    new_texts = [text]
    result, probabilitiy = predict(new_texts)
    print(f"The headline, \"{new_texts[0]}\" is:\n - {result}\n - {probabilitiy * 100:.2f}%")

In [34]:
predict_and_print("apple just ran out of money")

The headline, "apple just ran out of money" is:
 - neutral
 - 57.00%
